# Beschreibung: 

# Importe:

In [1]:
import sys
import os

import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))
from rst_functions import discretize, indiscernibility, dependency, quick_reduct, induce_rules, compute_coverage

os.getcwd()

'/home/samel/01. Projekte/01. Master/COMPARE_RST/Manuelle_Ausfuehrungen'

# Daten laden:

Student Performance & Behavior Dataset: https://www.kaggle.com/datasets/mahmoudelhemaly/students-grading-dataset?select=Students_Grading_Dataset_Biased.csv

In [2]:
unbiased = pd.read_csv("../noise_experiments/students_unbiased_noise_5.csv")

print(unbiased.shape)

(5000, 23)


# Ausführung

### Vorbereitung: (Datenaufbereitung)

In [14]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "Grade"

In [15]:
cutoffs = {
    "age": [20, 22], # Aufteilung in 18/19, 20/21, 22/23/24
    "Sleep_Hours_per_Night": [6, 8], # Aufteilung in 4/5, 6/7, 8/9  -  wenig, Durchschnitt, viel
    "Total_Score":[60, 70, 80, 90]  # Wichtig: Übliche Vergabe von Noten F: 0-60), D: 60-70), C: 70-80), B: 80-90), A: 90-100)
}

# Diskretisierung der numerischen Daten:
unbiased_disc = discretize(unbiased, bins=4,cutoffs=cutoffs)

# „Pass“ wieder anfügen, damit es NICHT diskretisiert wird.
unbiased_disc["Pass"] = (unbiased["Grade"].isin(["A", "B", "C", "D"])).astype(int)

In [16]:
# Konditionsattribute (Alle Werte - bis auf Student_ID und Email: conditional attributs all)
cond_attrs = []
for col in unbiased_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte (z. B. Total_Score_disc)
        if col.endswith("_disc"):
            cond_attrs.append(col)
        # direkt kategorische Werte (Gender, Department etc.)
        elif unbiased_disc[col].dtype == "object":
            cond_attrs.append(col)

cond_attrs.remove('Student_ID') # Das ist ein Identifier, daher muss es raus.
cond_attrs.remove('Email') # Es gilt hier dasselbe
# cond_attrs.remove('First_Name') # Das ist zwar kein Identifier, sorgt aber für einen sehr starke Streuung an Äquivalenzklassen. Im Beipsieldatensatz zwar nicht und kann daher da auch verwendet werden, aber sonst müsste daas raus.
# cond_attrs.remove('Last_Name') # Es gilt hier dasselbe

print(cond_attrs)

['First_Name', 'Last_Name', 'Gender', 'Department', 'Extracurricular_Activities', 'Internet_Access_at_Home', 'Parent_Education_Level', 'Family_Income_Level', 'Age_disc', 'Attendance (%)_disc', 'Midterm_Score_disc', 'Final_Score_disc', 'Assignments_Avg_disc', 'Quizzes_Avg_disc', 'Participation_Score_disc', 'Projects_Score_disc', 'Total_Score_disc', 'Study_Hours_per_Week_disc', 'Stress_Level (1-10)_disc', 'Sleep_Hours_per_Night_disc']


### Datenbetrachtung:

#### cond_attrs: Datensatz mit nicht nur diskretisierten numerischen Werten sondern auch direkt kategorische Werte (Gender, Department etc.)

In [17]:
# Redukte
unbiased_reduct, info_unbiased_grade = quick_reduct(unbiased_disc, cond_attrs, decision_attr)

print("\nUnbiased Reduct:\n", unbiased_reduct)

γ(C) mit allen Attributen: 0.795000
Einzel-γ-Werte:
  First_Name: γ = 0.000000
  Last_Name: γ = 0.000000
  Gender: γ = 0.000000
  Department: γ = 0.000000
  Extracurricular_Activities: γ = 0.000000
  Internet_Access_at_Home: γ = 0.000000
  Parent_Education_Level: γ = 0.000000
  Family_Income_Level: γ = 0.000000
  Age_disc: γ = 0.000000
  Attendance (%)_disc: γ = 0.000000
  Midterm_Score_disc: γ = 0.000000
  Final_Score_disc: γ = 0.000000
  Assignments_Avg_disc: γ = 0.000000
  Quizzes_Avg_disc: γ = 0.000000
  Participation_Score_disc: γ = 0.000000
  Projects_Score_disc: γ = 0.000000
  Total_Score_disc: γ = 0.000000
  Study_Hours_per_Week_disc: γ = 0.000000
  Stress_Level (1-10)_disc: γ = 0.000000
  Sleep_Hours_per_Night_disc: γ = 0.000000

Alle γ({a}) = 0, aber γ(C) > 0 → benutze quick_reduct_interaction.

Unbiased Reduct:
 ['First_Name', 'Total_Score_disc', 'Last_Name', 'Department']


In [18]:
rules_grade_unbiased = induce_rules(unbiased_disc, unbiased_reduct, decision_attr, verbose=True)

print("unbaised:")
for r in rules_grade_unbiased[:10]:
    print(r)

Anzahl an Klassen: 735

unbaised:
{'premise': {'First_Name': 'Omar', 'Total_Score_disc': np.int64(0), 'Last_Name': 'Williams', 'Department': 'Mathematics'}, 'decision': 'F', 'support': 2}
{'premise': {'First_Name': 'Omar', 'Total_Score_disc': np.int64(0), 'Last_Name': 'Williams', 'Department': 'Engineering'}, 'decision': 'F', 'support': 4}
{'premise': {'First_Name': 'Omar', 'Total_Score_disc': np.int64(0), 'Last_Name': 'Williams', 'Department': 'CS'}, 'decision': 'F', 'support': 2}
{'premise': {'First_Name': 'Omar', 'Total_Score_disc': np.int64(0), 'Last_Name': 'Brown', 'Department': 'Mathematics'}, 'decision': 'F', 'support': 2}
{'premise': {'First_Name': 'Omar', 'Total_Score_disc': np.int64(0), 'Last_Name': 'Brown', 'Department': 'Business'}, 'decision': 'F', 'support': 1}
{'premise': {'First_Name': 'Omar', 'Total_Score_disc': np.int64(0), 'Last_Name': 'Brown', 'Department': 'Engineering'}, 'decision': 'F', 'support': 1}
{'premise': {'First_Name': 'Omar', 'Total_Score_disc': np.int64

In [19]:
unbiased_pass_reduct, info_unbiased_pass = quick_reduct(unbiased_disc, cond_attrs, "Pass")

print("Unbiased Reduct:\n", unbiased_pass_reduct)

γ(C) mit allen Attributen: 0.795000
Einzel-γ-Werte:
  First_Name: γ = 0.000000
  Last_Name: γ = 0.000000
  Gender: γ = 0.000000
  Department: γ = 0.000000
  Extracurricular_Activities: γ = 0.000000
  Internet_Access_at_Home: γ = 0.000000
  Parent_Education_Level: γ = 0.000000
  Family_Income_Level: γ = 0.000000
  Age_disc: γ = 0.000000
  Attendance (%)_disc: γ = 0.000000
  Midterm_Score_disc: γ = 0.000000
  Final_Score_disc: γ = 0.000000
  Assignments_Avg_disc: γ = 0.000000
  Quizzes_Avg_disc: γ = 0.000000
  Participation_Score_disc: γ = 0.000000
  Projects_Score_disc: γ = 0.000000
  Total_Score_disc: γ = 0.591800
  Study_Hours_per_Week_disc: γ = 0.000000
  Stress_Level (1-10)_disc: γ = 0.000000
  Sleep_Hours_per_Night_disc: γ = 0.000000

Mindestens ein Attribut hat γ({a}) > 0 → benutze quick_reduct_monotone.
Unbiased Reduct:
 ['Total_Score_disc', 'First_Name', 'Last_Name']


In [20]:
rules_pass_unbiased = induce_rules(unbiased_disc, unbiased_pass_reduct, "Pass", verbose=True)

print("Unbaised:")
for r in rules_pass_unbiased[:10]:
    print(r)

Anzahl an Klassen: 207

Unbaised:
{'premise': {'Total_Score_disc': np.int64(0), 'First_Name': 'Omar', 'Last_Name': 'Williams'}, 'decision': np.int64(0), 'support': 8}
{'premise': {'Total_Score_disc': np.int64(0), 'First_Name': 'Omar', 'Last_Name': 'Brown'}, 'decision': np.int64(0), 'support': 5}
{'premise': {'Total_Score_disc': np.int64(0), 'First_Name': 'Omar', 'Last_Name': 'Jones'}, 'decision': np.int64(0), 'support': 4}
{'premise': {'Total_Score_disc': np.int64(0), 'First_Name': 'Omar', 'Last_Name': 'Smith'}, 'decision': np.int64(0), 'support': 8}
{'premise': {'Total_Score_disc': np.int64(0), 'First_Name': 'Omar', 'Last_Name': 'Davis'}, 'decision': np.int64(0), 'support': 2}
{'premise': {'Total_Score_disc': np.int64(0), 'First_Name': 'Omar', 'Last_Name': 'Johnson'}, 'decision': np.int64(0), 'support': 4}
{'premise': {'Total_Score_disc': np.int64(0), 'First_Name': 'Maria', 'Last_Name': 'Williams'}, 'decision': np.int64(0), 'support': 9}
{'premise': {'Total_Score_disc': np.int64(0), '

# Resultate

In [21]:
print("Ergebnisse:")

print("\nUnbiased:")
print("Pass:", unbiased_pass_reduct)
print("Pass Rules:", len(rules_pass_unbiased))
print("Grade:", unbiased_reduct)
print("Grade Rules:", len(rules_grade_unbiased))

Ergebnisse:

Unbiased:
Pass: ['Total_Score_disc', 'First_Name', 'Last_Name']
Pass Rules: 187
Grade: ['First_Name', 'Total_Score_disc', 'Last_Name', 'Department']
Grade Rules: 632


In [28]:
#ind = indiscernibility(biased_disc, biased_pass_reduct)
#ind
dependency(unbiased_disc, ['First_Name', 'Total_Score_disc', 'Last_Name', 'Department'], decision_attr)
dependency(unbiased_disc, ['Total_Score_disc', 'First_Name', 'Last_Name'], "Pass")

0.9252

# Wiedergabe der Abdeckung durch Regeln:
### Hier nur zum Bestehen

In [23]:
# Annahme: Folgendes muss vorhanden sein:
# unbiased_disc, unbiased_pass_reduct, rules_pass_unbiased

cov_unbiased = compute_coverage(
    unbiased_disc,
    unbiased_pass_reduct,
    "Pass",
    rules_pass_unbiased
)

print("Coverage unbiased:", cov_unbiased)

cov_unbiased_pct = cov_unbiased * 100

Coverage unbiased: 0.9252


### Hier für Note:

In [24]:
cov_unbiased = compute_coverage(
    unbiased_disc,
    unbiased_reduct,
    "Grade",
    rules_grade_unbiased
)

print("Coverage grade unbiased:", cov_unbiased)

cov_unbiased_pct = cov_unbiased * 100

Coverage grade unbiased: 0.8322
